# Task 4 — LSTM: Gate-by-Gate Breakdown

### Introduction
To solve the vanishing gradient problem, Hochreiter & Schmidhuber introduced the **Long Short-Term Memory (LSTM)** in 1997.

#### The Shared Notebook Analogy
If a standard RNN is like a telephone game where messages fade away, an LSTM is like a **shared notebook** (the **cell state**, $C_t$). As each person reads, they can write notes, cross things out, or read notes from the page, but the page itself is passed directly from start to finish. This direct conveyor belt allows information to flow backward through time without fading.

To manage this notebook, LSTMs use three mathematical **gates**:
1. **Forget Gate** ($f_t$): Decides what information to cross out and erase from the notebook.
2. **Input Gate** ($i_t$): Decides what new information to write on the page.
3. **Output Gate** ($o_t$): Decides what information to read from the page to produce the next hidden state $h_t$ (the mental summary).

---

### Conceptual Diagrams: LSTM Cell Architecture
Below is the inner structure of a Long Short-Term Memory (LSTM) cell:

![LSTM chain](outputs/LSTM-chain.png)

#### 1. The Cell State Conveyor Belt ($C_t$)
Look at the horizontal line running along the top of the LSTM cell diagram:
- This is the **Cell State** ($C_t$). It runs straight down the entire chain with only minor linear interactions.
- Because information flows down this belt with only simple addition and multiplication operations, gradients can propagate backward through time along this line with **very little decay**. This is the key mechanism that solves the vanishing gradient problem!

#### 2. Detailed Gate breakdowns & Gate Equations:

*   **Forget Gate ($f_t$)** (Visual: `outputs/LSTM-forget.png`):
    - **Purpose**: Decides what information from the cell state should be discarded (like a trash bin filter).
    - **Formula**: $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
    - **Explanation**: It takes the previous hidden state $h_{t-1}$ and current input $x_t$, passes them through a sigmoid function ($\sigma$). A sigmoid output of `1` means "completely keep this", while `0` means "completely forget this".

*   **Input Gate ($i_t$) & Candidate State ($\tilde{C}_t$)** (Visual: `outputs/LSTM-input.png`):
    - **Purpose**: Decides which new information should be added to the cell state.
    - **Formulas**: 
      - $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ (Gating vector)
      - $\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$ (Candidate vector of new values)
    - **Explanation**: The sigmoid input gate $i_t$ decides *which* values to update. The $\tanh$ candidate state $\tilde{C}_t$ creates a vector of *new candidate values* that could be added to the cell state.

*   **Updating the Cell State ($C_t$)**:
    - **Formula**: $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$
    - **Explanation**: We scale the old state by the forget gate ($f_t * C_{t-1}$) to drop unwanted history, then add the new candidate state scaled by the input gate ($i_t * \tilde{C}_t$).

*   **Output Gate ($o_t$) & Hidden State ($h_t$)** (Visual: `outputs/LSTM-output.png`):
    - **Purpose**: Decides what the next hidden state $h_t$ (the exposed output of the cell) should be.
    - **Formulas**:
      - $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
      - $h_t = o_t * \tanh(C_t)$
    - **Explanation**: The output gate sigmoid $o_t$ decides what parts of the cell state we are going to output. Then, we push the cell state through a $\tanh$ activation (to squeeze the values between $[-1, 1]$) and multiply it by the output gate $o_t$.

### Core Use Cases
- **Sentence Information Conservation**: Ensuring subject details from the beginning of a paragraph are not lost.
- **Coreference Resolution**: Mapping pronouns back to correct nouns across long context spans.



### Step 1 — Implement an LSTM Cell Step from Scratch


In [1]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

class LSTMCellScratch:
    def __init__(self, input_dim, hidden_dim):
        np.random.seed(42)
        self.hidden_dim = hidden_dim
        self.W = np.random.randn(4 * hidden_dim, input_dim + hidden_dim) * 0.1
        self.b = np.zeros((4 * hidden_dim, 1))
        
    def forward(self, x_t, h_prev, c_prev):
        concat = np.vstack((h_prev, x_t))
        gates = np.dot(self.W, concat) + self.b
        
        f_gate = gates[0:self.hidden_dim]
        i_gate = gates[self.hidden_dim:2*self.hidden_dim]
        c_tilde = gates[2*self.hidden_dim:3*self.hidden_dim]
        o_gate = gates[3*self.hidden_dim:]
        
        f_t = sigmoid(f_gate)
        i_t = sigmoid(i_gate)
        c_tilde_t = np.tanh(c_tilde)
        o_t = sigmoid(o_gate)
        
        c_t = f_t * c_prev + i_t * c_tilde_t
        h_t = o_t * np.tanh(c_t)
        
        return h_t, c_t, {
            'forget_gate': f_t,
            'input_gate': i_t,
            'candidate_cell': c_tilde_t,
            'output_gate': o_t
        }

# Instantiate cell
input_dim = 3
hidden_dim = 2
cell = LSTMCellScratch(input_dim, hidden_dim)

# Input parameters
x = np.array([[1.0, -0.5, 0.2]]).T
h_p = np.array([[0.1, -0.2]]).T
c_p = np.array([[0.5, 0.5]]).T

h_n, c_n, gate_vals = cell.forward(x, h_p, c_p)

print("LSTM Cell Forward Pass Outputs:")
print(f"  Forget Gate Output (f_t)   :\n{gate_vals['forget_gate']}")
print(f"  Input Gate Output (i_t)    :\n{gate_vals['input_gate']}")
print(f"  Candidate Cell State (~C_t):\n{gate_vals['candidate_cell']}")
print(f"  Updated Cell State (C_t)   :\n{c_n}")
print(f"  Output Gate Output (o_t)   :\n{gate_vals['output_gate']}")
print(f"  New Hidden State (h_t)     :\n{h_n}")



LSTM Cell Forward Pass Outputs:
  Forget Gate Output (f_t)   :
[[0.49791669]
 [0.51927613]]
  Input Gate Output (i_t)    :
[[0.52249538]
 [0.51579814]]
  Candidate Cell State (~C_t):
[[0.08606117]
 [0.08568632]]
  Updated Cell State (C_t)   :
[[0.29392491]
 [0.30383491]]
  Output Gate Output (o_t)   :
[[0.5062307 ]
 [0.46455799]]
  New Hidden State (h_t)     :
[[0.14465202]
 [0.13696013]]


### Step 2 — User-Defined Interactive Evaluation Function
Provide custom values for the inputs and previous states to calculate LSTM gate responses dynamically.


In [2]:
def evaluate_lstm_gates(input_vals, forget_bias=0.0):
    """
    User-defined evaluation function. Simulates LSTM gate logic under custom parameters.
    """
    x_input = np.array([input_vals]).T
    h_p = np.zeros((hidden_dim, 1))
    c_p = np.ones((hidden_dim, 1)) * 0.8
    
    temp_cell = LSTMCellScratch(len(input_vals), hidden_dim)
    temp_cell.b[0:hidden_dim] += forget_bias
    
    h_n, c_n, gates = temp_cell.forward(x_input, h_p, c_p)
    
    print(f"--- Evaluation for Input Vector: {input_vals} ---")
    print(f"Forget Bias Offset: {forget_bias}")
    print("-" * 50)
    print(f"Forget Gate values  : {gates['forget_gate'].flatten().round(4)}")
    print(f"Input Gate values   : {gates['input_gate'].flatten().round(4)}")
    print(f"Candidate Cell State: {gates['candidate_cell'].flatten().round(4)}")
    print(f"Updated Cell State  : {c_n.flatten().round(4)}")
    print(f"Output Gate values  : {gates['output_gate'].flatten().round(4)}")
    print(f"New Hidden State    : {h_n.flatten().round(4)}")

# Test with high forget bias (remembers old cell state)
evaluate_lstm_gates([1.0, 2.0, -1.0], forget_bias=5.0)
print("\n" + "="*60 + "\n")
# Test with low forget bias (wipes old cell state)
evaluate_lstm_gates([1.0, 2.0, -1.0], forget_bias=-5.0)



--- Evaluation for Input Vector: [1.0, 2.0, -1.0] ---
Forget Bias Offset: 5.0
--------------------------------------------------
Forget Gate values  : [0.9955 0.9928]
Input Gate values   : [0.4536 0.4978]
Candidate Cell State: [-0.2201 -0.0533]
Updated Cell State  : [0.6965 0.7677]
Output Gate values  : [0.4267 0.3819]
New Hidden State    : [0.257  0.2466]


--- Evaluation for Input Vector: [1.0, 2.0, -1.0] ---
Forget Bias Offset: -5.0
--------------------------------------------------
Forget Gate values  : [0.0099 0.0062]
Input Gate values   : [0.4536 0.4978]
Candidate Cell State: [-0.2201 -0.0533]
Updated Cell State  : [-0.0919 -0.0216]
Output Gate values  : [0.4267 0.3819]
New Hidden State    : [-0.0391 -0.0082]
